# Circuit Identities and Simplification

## Circuits with the same overall unitary can look very different on paper. A handful of identities let you simplify, compile, and optimise circuits — they are also a great way to develop intuition for what gates *do*.

# 1. Single-qubit identities

## - $X^2 = Y^2 = Z^2 = H^2 = I$ — these are all self-inverse.
## - $X = HZH$, $Z = HXH$, $Y = -HYH$ — Hadamard swaps $X$ and $Z$.
## - $S = \sqrt{Z}$, $T = \sqrt{S}$: $S^2 = Z$, $T^2 = S$, $T^4 = Z$.
## - $XZ = -ZX$, $XY = iZ$, etc. (the Pauli algebra).

In [1]:
import numpy as np

X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)
H = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)

print('HZH = X ?', np.allclose(H @ Z @ H, X))
print('HXH = Z ?', np.allclose(H @ X @ H, Z))
print('HYH = -Y?', np.allclose(H @ Y @ H, -Y))

HZH = X ? True
HXH = Z ? True
HYH = -Y? True


# 2. CNOT identities

## Let $\text{CNOT}_{ij}$ mean control $i$, target $j$. Then:

## - $\text{CNOT}_{ij}^2 = I$.
## - $\text{CNOT}_{ij}\,(X_i)\,\text{CNOT}_{ij} = X_i X_j$ — $X$ on the control propagates to the target.
## - $\text{CNOT}_{ij}\,(Z_j)\,\text{CNOT}_{ij} = Z_i Z_j$ — $Z$ on the target propagates to the control.
## - $\text{SWAP}_{ij} = \text{CNOT}_{ij}\,\text{CNOT}_{ji}\,\text{CNOT}_{ij}$ (three CNOTs).
## - $\text{CNOT}_{ij} = (I \otimes H)\,\text{CZ}_{ij}\,(I \otimes H)$ — switch between CNOT and CZ via Hadamards on the target.

In [2]:
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
CZ   = np.diag([1,1,1,-1]).astype(complex)
I    = np.eye(2, dtype=complex)

lhs = np.kron(I, H) @ CZ @ np.kron(I, H)
print('(I o H) CZ (I o H) = CNOT ?', np.allclose(lhs, CNOT))

(I o H) CZ (I o H) = CNOT ? True


# 3. Cancelling adjacent inverses

## The most common simplification: $U^{-1}\,U = I$. So a circuit fragment like $H \cdot X \cdot H \cdot H \cdot X \cdot H$ collapses to nothing. Compilers scan the circuit and remove such fragments first.

# 4. Moving gates past each other

## Two gates that act on **disjoint qubits** always commute: their order can be swapped freely. Two gates on the same qubit only commute when their matrix product is the same in both orders — for example, $Z$ commutes with itself and with $T$ (both diagonal), but does not commute with $X$.

## This is the basis of *commutation-based* circuit optimisation: shuffle gates to expose cancellations.

In [3]:
# Show that a Z gate on q0 commutes with an arbitrary gate on q1
Z0 = np.kron(Z, I)
A1 = np.kron(I, np.array([[0.6, 0.8],[0.8, -0.6]], dtype=complex))
print('[Z_0, A_1] = 0 ?', np.allclose(Z0 @ A1, A1 @ Z0))

[Z_0, A_1] = 0 ? True


# 5. Useful compound identities

## - $(H \otimes H)\,\text{CNOT}_{01}\,(H \otimes H) = \text{CNOT}_{10}$ — wrap CNOT in Hadamards on both qubits to swap control and target.
## - $(I \otimes X)\,\text{CNOT}\,(I \otimes X) = \text{CNOT}\,(I \otimes X)\,(I \otimes X) = \text{CNOT}$ — $X$ on the target commutes with CNOT.
## - An $R_z(\theta)$ on the target *commutes* with the CNOT control because $R_z$ is diagonal.

# 6. Why this matters in practice

## Real hardware has a tiny native gate set and short coherence times. A good compiler:

## - Decomposes high-level gates into native ones.
## - Cancels redundant gates.
## - Schedules in parallel to minimise depth.
## - Maps logical qubits to physical ones, adding SWAPs only where necessary.

## Every line of optimisation directly reduces decoherence — which is the main source of error today.

# Recap

## - A handful of identities let you rewrite circuits without changing their action.
## - $HZH = X$, $\text{CNOT}^2 = I$, $\text{SWAP} = 3 \text{ CNOTs}$ are the workhorses.
## - Disjoint-qubit gates commute; this is what allows depth optimisation.

## Next: stop hand-rolling matrices and start using Qiskit.